In [ ]:
import warnings
warnings.filterwarnings("ignore")
!pip install scikeras
!apt-get install perl
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from scikeras.wrappers import KerasClassifier
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression

In [ ]:
from google.colab import drive
#drive.mount('/content/gdrive')
drive.mount("/content/gdrive", force_remount=True)

In [ ]:
def read_file(file_path):
    with open(file_path, "r") as file:
        cleaned_lines = file.readlines()
        #cleaned_lines = [line.split("#")[0].strip() for line in lines]

    data = []
    for line in cleaned_lines:
        row = {}
        parts = line.split()
        row['Label'] = parts[0]
        for part in parts[1:]:
            key, value = part.split(":")
            if key == 'qid':
                row[key] = value
            else:
                row[f"F_{key}"] = float(value)# if value != "NULL" else -1
        data.append(row)

    return pd.DataFrame(data)

df_fs = read_file("/content/gdrive/MyDrive/MSLR-Web10K/Fold1/test.txt")
print(df_fs.shape)
#print(df_fs.columns[30])
#print(df_fs[df_fs.columns[29]])

"""
TF, IDF, TF_IDF, TF_Title, IDF_Title, TF_IDF_Title, TF_URL, IDF_URL, TF_IDF_URL, DL, DL_Title, DL_URL, BM25, PageRank, HITS_Hub, HITS_Authority,
First_Session, Last_Session, doc_clicks_query, doc_sessions_clicks_query, doc_all_clicks, sessions_clicked, queries_clicked, single_clciks_sessions,
single_clciks_queries, abs_single_clciks_queries, single_clciks_grouped_sessions, non_single_clciks_sessions, non_single_clciks_queries
"""

Normalize DataFrame of data

In [ ]:
import pandas as pd

# Assuming your dataframe is named 'df'
def normalize_columns(df):
    """Normalizes columns F_1 to F_29 in a dataframe.

    Args:
        df: The dataframe to normalize.

    Returns:
        The normalized dataframe.
    """

    for col in df.columns[2:]:  # Start from F_1 (index 3)
        if col.startswith('F_'):
            df[col] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())
    return df

normalized_df_fs = normalize_columns(df_fs)
print(normalized_df_fs.columns)
print(normalized_df_fs)

Calculate Kendall Tau matrix of all data

In [ ]:
import pandas as pd
from scipy.stats import kendalltau
import numpy as np

def create_kendall_tau_matrix(df, features):
    """
    Calculates the Kendall Tau correlation matrix for the specified features.

    Args:
        df: The DataFrame containing the data.
        features: A list of feature names to include in the matrix.

    Returns:
        A NumPy array representing the Kendall Tau correlation matrix.
    """

    n_features = len(features)
    kendall_tau_matrix = np.zeros((n_features, n_features))

    for i in range(n_features):
        for j in range(i + 1, n_features):
            feature_i = features[i]
            feature_j = features[j]

            # Group by 'qid' and calculate Kendall's Tau for each group
            kendall_tau_values = df.groupby('qid').apply(lambda x: kendalltau(x[feature_i], x[feature_j])[0])

            # Calculate the average Kendall's Tau
            avg_kendall_tau = kendall_tau_values.mean()

            # Fill in the matrix symmetrically
            kendall_tau_matrix[i, j] = avg_kendall_tau
            kendall_tau_matrix[j, i] = avg_kendall_tau

    return kendall_tau_matrix

# Specify the desired features
features = ['Label', 'F_134', 'F_135', 'F_136']

# Calculate the Kendall Tau matrix for the specified features
kendall_tau_matrix = create_kendall_tau_matrix(normalized_df_fs, features)
kendall_tau_matrix[np.isnan(kendall_tau_matrix)] = 0
print(kendall_tau_matrix.shape)
pruning_threshold = 0.12
kendall_tau_matrix_pruned_cf  = np.where(kendall_tau_matrix < pruning_threshold, 0, kendall_tau_matrix)
print(kendall_tau_matrix_pruned_cf)

In [ ]:
print(kendall_tau_matrix_pruned_cf)
kendall_tau_matrix_pruned_cf_1 = kendall_tau_matrix_pruned_cf
#kendall_tau_matrix_pruned_cf_1[0, 1:] += 0.67
kendall_tau_matrix_pruned_cf_1[1:, 0] += 0.67
print(kendall_tau_matrix_pruned_cf_1)

In [ ]:
kendall_tau_matrix_pruned_cf_1[0, 1:] += 0.67
print(kendall_tau_matrix_pruned_cf_1)

**Step#1: Check relationship between CFs and Relevance by calculating Kendall-Tau Matrix of CFs**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import files
import shutil

# Assuming kendall_tau_matrix is your calculated Kendall Tau matrix

# Set up the figure with a clear title and labels
plt.figure(figsize=(3, 3), dpi=1200)
#plt.title("Kendall-Tau Correlation Heatmap for Rankers")
#plt.xlabel("Relevance and CTR Features")
#plt.ylabel("Relevance and CTR Features")

# Define the labels for the horizontal and vertical axes

#labels = ["Relevance", "TF", "IDF", "TF_IDF", "TF_Title", "IDF_Title", "TF_IDF_Title", "TF_URL", "IDF_URL", "TF_IDF_URL", "DL", "DL_Title", "DL_URL", "BM25", "PageRank", "HITS_Hub", "HITS_Authority", "First_Session", "Last_Session", "doc_clicks_query", "doc_sessions_clicks_query",
#          "doc_all_clicks", "sessions_clicked", "queries_clicked", "single_clicks_sessions", "single_clicks_queries",
#          "abs_single_clicks_queries", "single_clicks_grouped_sessions", "non_single_clicks_sessions", "non_single_clicks_queries"]

#labels = ["Relevance", "First_Session", "Last_Session", "doc_clicks_query", "doc_sessions_clicks_query",
#          "doc_all_clicks", "sessions_clicked", "queries_clicked", "single_clicks_sessions", "single_clicks_queries",
#          "abs_single_clicks_queries", "single_clicks_grouped_sessions", "non_single_clicks_sessions", "non_single_clicks_queries"]

#labels = ["Relevance", "First_Session", "Last_Session", "doc_clicks_query", "doc_sessions_clicks_query",
#          "doc_all_clicks", "sessions_clicked", "queries_clicked", "single_clicks_sessions",
#          "abs_single_clicks_queries", "single_clicks_grouped_sessions", "non_single_clicks_sessions", "non_single_clicks_queries"]

labels = ["Relevance", "F134", "F135", "F136"]

# Create the heatmap with adjusted parameters and meaningful annotations
heatmap = sns.heatmap(
    kendall_tau_matrix_pruned_cf,
    annot=True,  # Show cell texts
    #annot=False,  # Hide cell texts
    fmt=".2f",
    cmap="YlGnBu",  # Consider a colormap that better differentiates values
    vmin=-1,
    vmax=1,
    xticklabels=labels,
    yticklabels=labels
)

# Adjust annotations for clarity and make font bold
for text in heatmap.texts:
    text.set_ha("center")  # Center-align text
    text.set_va("center")  # Center-align text vertically
    text.set_weight("bold")  # Make font bold
    text.set_fontsize(10)  # Decrease font size

# Save the figure locally with a descriptive filename
outputFileName = f"KendallTau_correlation_heatmap_MSLR_pruning_{pruning_threshold}.png"
plt.savefig(outputFileName, dpi=1200, bbox_inches='tight')

# Save the figure
# Download the figure to the specified directory
#local_path = 'D:\\Me\\Teaching\\4023\\RD\\DeepRankAgg\\Figs'
#shutil.copy('heatmap.png', local_path)

**Step#2: Assign weights to all features based on:**
1.   Retrieval performance of features (P@1)
2.   Kendall Tau of features and relevance

Weight_1: Kendall Tau of features and relevance

In [ ]:
kendall_weights = kendall_tau_matrix_pruned[1:, 0]
kendall_weights_normalized = (kendall_weights - kendall_weights.min()) / (kendall_weights.max() - kendall_weights.min())
print(kendall_weights_normalized.shape)

Weight_2: P@1 of features

In [ ]:
fnames = ["TF", "IDF", "TF_IDF", "TF_Title", "IDF_Title", "TF_IDF_Title", "TF_URL", "IDF_URL", "TF_IDF_URL", "DL", "DL_Title", "DL_URL", "BM25", "PageRank", "HITS_Hub", "HITS_Authority",
          "First_Session", "Last_Session", "doc_clicks_query", "doc_sessions_clicks_query", "doc_all_clicks", "sessions_clicked", "queries_clicked", "single_clciks_sessions",
          "single_clciks_queries", "abs_single_clciks_queries", "single_clciks_grouped_sessions", "non_single_clciks_sessions", "non_single_clciks_queries"]
wcl2r_features_names = np.array(fnames)

!cp /content/gdrive/MyDrive/WCL2R/mslr-eval-score-mslr.pl /content/

with open("/content/gdrive/MyDrive/WCL2R/FS.txt", "r") as infile, open(f"/content/judgements.txt", "w") as outfile:
    for line in infile:
        parts = line.split()
        label = parts[0]
        qid_part = parts[1]
        #docid_part = parts[-1]
        #inc_part = parts[-4]
        #prob_part = parts[-1]

        #outfile.write(f"{parts[0]} qid:{qid_part} #docid = {docid_part} inc = {inc_part} prob = {prob_part}\n")
        outfile.write(f"{parts[0]} {qid_part}\n")

n_features = 31 #features_list + qid
# Initialize lists to store evaluation metrics
#p_at_1 = pd.Series(np.zeros(29))
p_at_1 = pd.Series(np.zeros(29))
print(p_at_1.shape)

for i in range(2, n_features):
    predicted_labels = normalized_df_fs[normalized_df_fs.columns[i]]
    print(predicted_labels)
    with open(f'predictions-F1.txt', 'w') as f:
        for f1 in predicted_labels:
            f.write(f"  {f1:.7e}  \n")

    # Evaluate final predictions using external script
    !perl mslr-eval-score-mslr.pl judgements.txt predictions-F1.txt output.txt 0
    print(f"FINAL RESULT :\n")
    with open(f"/content/output.txt", "r") as final_output:
        print(final_output.read())

    # Read and store evaluation metrics
    df = pd.read_csv('/content/output.txt', delimiter='\t', header=None)
    #p_at_1[i].append(float(df.iloc[1, 1]))
    p_at_1[i-2] = float(df.iloc[1, 1])

print(p_at_1.shape)
p_at_1_normalized = (p_at_1 - p_at_1.min()) / (p_at_1.max() - p_at_1.min())
print(p_at_1_normalized.shape)
print(p_at_1_normalized)

Integrate weight_1 and weight_2: [kendall_weights_normalized, p_at_1_normalized]

In [ ]:
weights = np.column_stack((kendall_weights_normalized, p_at_1_normalized))
print(weights)

# Sort the first 16 rows based on the second column in descending order
top_16_rows = weights[:16, :][np.argsort(weights[:16, 1])[::-1]]

# Get the indices of the sorted rows
tops_NC = np.argsort(weights[:16, 1])[::-1] + 1
print(tops_NC)

# Calculate the index of the 13th last row
last_13_start_index = len(weights) - 13

# Sort the last 13 rows based on the second column in descending order
last_13_sorted_rows = weights[last_13_start_index:, :][np.argsort(weights[last_13_start_index:, 1])[::-1]]

# Get the indices of the sorted rows within the last 13 rows
last_13_indices = np.argsort(weights[last_13_start_index:, 1])[::-1] + last_13_start_index

# Assign the indices to the 'tops_NC' array
tops_CF = last_13_indices + 1
print(tops_CF)

**Step#3: Build features similarity graph between NCs and Cluster Them**

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm


def create_FSG_graph(matrix):
    """
    Creates an RSG graph from the given similarity matrix.

    Args:
        matrix: A NumPy array representing the similarity matrix.

    Returns:
        A networkx.Graph object representing the RSG.
    """

    G = nx.Graph()
    for i in range(len(matrix)):
        for j in range(i + 1, len(matrix)):
            if matrix[i][j] != 0:
                G.add_edge(i + 1, j + 1, weight=matrix[i][j])
    return G


def draw_FSG_graph(G, grayscale=False, layout_algo='kamada_kawai', **kwargs):
    """
    Draws the RSG graph with varying edge thickness based on weight.

    Args:
        G: A networkx.Graph object representing the RSG.
        grayscale (bool, optional): If True, draws the graph in grayscale. Defaults to False.
        layout_algo (str, optional): The layout algorithm to use. Defaults to 'kamada_kawai'.
        **kwargs: Additional keyword arguments to be passed to the chosen layout algorithm.

    Returns:
        None (plots the graph directly).
    """

    # Use a force-directed layout for better node positioning, with customization
    if layout_algo == 'kamada_kawai':
        pos = nx.kamada_kawai_layout(G, **kwargs)
    elif layout_algo == 'spring':
        pos = nx.spring_layout(G, **kwargs)
    else:
        raise ValueError(f"Invalid layout algorithm '{layout_algo}': Choose from 'kamada_kawai' or 'spring'")

    # Adjust node size and spacing for readability
    node_size = 250
    node_spacing = 2.0

    # Create a colormap based on edge weights
    cmap = cm.get_cmap('viridis')  # Adjust colormap as needed

    # Draw nodes and labels
    nx.draw_networkx_nodes(G, pos, node_size=node_size, node_color='lightblue')
    nx.draw_networkx_labels(G, pos, font_size=12, font_family='sans-serif')

    # Draw edges with varying thickness and color
    edges = G.edges()
    weights = [G[u][v]['weight'] for u, v in edges]
    if grayscale:
        colors = cm.gray(weights)
    else:
        colors = cmap(weights)
    nx.draw_networkx_edges(G, pos, width=weights, edge_color=colors, arrowsize=10)

    # Adjust figure size and margins
    plt.figure(figsize=(30, 30), dpi=1200)
    plt.margins(0.2)

    plt.axis('off')

    plt.tight_layout()
    plt.show()

NC_FSG = create_FSG_graph(kendall_tau_matrix_pruned_nc)

# Option 1: Default layout (kamada_kawai) with some customization
#draw_FSG_graph(NC_FSG, grayscale=False)

# Option 2: Spring layout with node repel parameter
#draw_FSG_graph(NC_FSG, grayscale=False, layout_algo='spring', k=0.5)  # Adjust k for node repulsion

In [ ]:
# Install the louvain package if not already installed
import numpy as np
from sklearn.cluster import SpectralClustering
from sklearn.metrics import silhouette_score

def find_best_num_clusters(graph_adjacency_matrix, num_clusters_range):
    """
    Automatically finds the best number of clusters for a given weighted graph using spectral clustering.

    Args:
        graph_adjacency_matrix: The adjacency matrix of the graph.
        num_clusters_range: The range of cluster numbers to explore.

    Returns:
        The best number of clusters and the corresponding cluster labels.
    """

    best_num_clusters = None
    best_silhouette_score = -1

    for num_clusters in num_clusters_range:
        spectral_clustering = SpectralClustering(n_clusters=num_clusters, affinity='precomputed')
        cluster_labels = spectral_clustering.fit_predict(graph_adjacency_matrix)

        silhouette_avg = silhouette_score(graph_adjacency_matrix, cluster_labels)

        if silhouette_avg > best_silhouette_score:
            best_num_clusters = num_clusters
            best_cluster_labels = cluster_labels
            best_silhouette_score = silhouette_avg

    return best_num_clusters, best_cluster_labels

num_clusters_range = range(2, 5)  # Explore clusters from 2 to 10

best_num_clusters, cluster_labels = find_best_num_clusters(kendall_tau_matrix_pruned_nc, num_clusters_range)

print("Best number of clusters:", best_num_clusters)
print("Cluster labels:", cluster_labels)

Get top **k%** of **top non-click features** of each cluster based on their P@1 weights

In [ ]:
print("Cluster labels:", cluster_labels)
print(weights[:16, 1])
# Assuming 'weights' is a 2D NumPy array and 'cluster_labels' is a 1D NumPy array
# Sort the first 16 rows based on the second column in descending order
top_16_rows = weights[:16, :][np.argsort(weights[:16, 1])[::-1]]

# Get the indices of the sorted rows
sorted_indices = np.argsort(weights[:16, 1])[::-1]

# Define the desired percentage (e.g., 60%)
k_NC_percent = 60

# Initialize a dictionary to store top indices for each cluster
top_indices_per_cluster = {}

# Iterate over unique cluster labels
for cluster_label in np.unique(cluster_labels):
    # Get indices of rows belonging to the current cluster
    cluster_indices = np.where(cluster_labels[:16] == cluster_label)[0]

    # Get the corresponding rows from the sorted 16 rows
    cluster_sorted_rows = top_16_rows[cluster_indices]

    # Sort the cluster-specific rows based on their second column
    cluster_sorted_indices = np.argsort(cluster_sorted_rows[:, 1])[::-1]

    # Calculate the number of top rows for the current cluster based on its size
    num_top_rows_per_cluster = int(k_NC_percent / 100 * len(cluster_indices))

    # Get the top indices within the cluster, relative to the original cluster indices
    top_cluster_indices = cluster_indices[cluster_sorted_indices[:num_top_rows_per_cluster]]

    # Store the top indices for the current cluster
    top_indices_per_cluster[cluster_label] = top_cluster_indices + 1  # Add 1 to start from 1

# Print the top indices for each cluster
for cluster_label, indices in top_indices_per_cluster.items():
    print(f"Cluster {cluster_label}: {indices}")

tops_NC = []
for cluster_label, indices in top_indices_per_cluster.items():
    tops_NC.extend(indices)

print(tops_NC)

Get top **k%** of **top click features** of each cluster based on their P@1 weights

In [ ]:
# Sort the last 13 rows based on the second column in descending order
last_13_rows = weights[-13:, :][np.argsort(weights[-13:, 1])[::-1]]

# Get the indices of the sorted rows
sorted_indices = np.argsort(weights[-13:, 1])[::-1] + len(weights) - 13

# Define the desired percentage (e.g., 35%)
k_CF_percent = 35

# Initialize an empty list to store the top indices
tops_CF = []

# Iterate over unique cluster labels
for cluster_label in np.unique(cluster_labels[-13:]):
    # Get indices of rows belonging to the current cluster
    cluster_indices = np.where(cluster_labels[-13:] == cluster_label)[0] + len(weights) - 13

    # Get the corresponding rows from the sorted 13 rows
    cluster_sorted_rows = last_13_rows[cluster_indices - (len(weights) - 13)]

    # Sort the cluster-specific rows based on their second column
    cluster_sorted_indices = np.argsort(cluster_sorted_rows[:, 1])[::-1]

    # Calculate the number of top rows for the current cluster based on its size
    num_top_rows_per_cluster = int(k_CF_percent / 100 * len(cluster_indices))

    # Get the top indices within the cluster, relative to the original cluster indices
    top_cluster_indices = cluster_indices[cluster_sorted_indices[:num_top_rows_per_cluster]] + 1

    # Append the top indices to the tops_CF list
    tops_CF.extend(top_cluster_indices.tolist())

# Print the top indices for all clusters
print(tops_CF)

**Step#4: For each of top CF features, create a deep model to predict it using top NC features**

In [ ]:
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import random
from tensorflow.keras.utils import plot_model
import graphviz  # Import graphviz
from tensorflow.keras.models import load_model
warnings.filterwarnings("ignore")

def create_model(input_shape, num_layers, num_neurons):
    # Initialize a sequential model
    model = Sequential()
    # Add the first dense layer with L2 regularization and a dropout layer
    model.add(Dense(num_neurons, activation='relu', input_shape=(input_shape,), kernel_regularizer=l2(0.008))) #0.008
    model.add(Dropout(0.5))

    # Add additional layers as specified by num_layers
    for _ in range(num_layers - 1):
        model.add(Dense(num_neurons, activation='relu', kernel_regularizer=l2(0.008)))
        model.add(Dropout(0.5))

    # Add the output layer
    model.add(Dense(1, activation='sigmoid'))

    # Compile the model with Adam optimizer and mean squared error loss
    #model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
    model.compile(optimizer='rmsprop', loss='mean_squared_error', metrics=['acc'])

    return model

print(tops_CF)
print(tops_NC)

# Number of iterations for random search
num_iterations = 2
count_of_models = num_iterations

# Define hyperparameter ranges
num_layers_range = [2, 3, 4]
num_neurons_range = [4, 8, 16, 32, 64, 128, 256]
batch_size_range = [4, 8, 16, 32, 64, 128, 256]

for cf_index in tops_CF:
    X_train, y_train = normalized_df_fs[['F_' + str(i) for i in tops_NC]].astype(float), normalized_df_fs[f'F_{cf_index}'].astype(float)
    X_test, y_test = normalized_df_fs[['F_' + str(i) for i in tops_NC]].astype(float), normalized_df_fs[f'F_{cf_index}'].astype(float)
    X_vali, y_vali = normalized_df_fs[['F_' + str(i) for i in tops_NC]].astype(float), normalized_df_fs[f'F_{cf_index}'].astype(float)
    # Initialize variables
    best_config = None
    best_val_loss = float('inf')
    val_losses = []
    num_layers_history = []
    num_neurons_history = []
    batch_size_history = []

    # Scale the training and validation data and test data
    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_vali_scaled = scaler_X.transform(X_vali)
    X_test_scaled = scaler_X.transform(X_test)

    # Initialize a MinMaxScaler for later use
    scaler = MinMaxScaler(feature_range=(0, 2))

    # Initialize lists to store predictions
    predictions = []
    train_predictions = []
    test_predictions = []
    for iterat in range(num_iterations):
        # Randomly sample hyperparameters
        num_layers = random.choice(num_layers_range)
        num_neurons = random.choice(num_neurons_range)
        bs = random.choice(batch_size_range)
        print(" **** Model Config[",iterat,"]: num_layers=", num_layers, ", num_neurons=", num_neurons, ", batch_size=", bs)

        # Create and train the model
        model = create_model(X_train_scaled.shape[1], num_layers, num_neurons)
        early_stopping = EarlyStopping(monitor='val_loss', patience=1, restore_best_weights=True)
        history = model.fit(X_train_scaled, y_train, epochs=6, batch_size=bs, validation_data=(X_vali_scaled, y_vali), callbacks=[early_stopping], verbose=0)

        # Record hyperparameters and validation loss
        val_loss = history.history['val_loss'][-1]

        # Update best configuration if necessary
        if val_loss < best_val_loss:
            best_config = (num_layers, num_neurons, bs)
            best_val_loss = val_loss
            # Optionally, save the best model for later use
            #model.save("best_model.h5")
            model.save_weights("best_model.weights.h5")

        val_losses.append(val_loss)
        num_layers_history.append(num_layers)
        num_neurons_history.append(num_neurons)
        batch_size_history.append(bs)

    # Print the best configuration
    print("Best Configuration:", best_config)
    num_layers = best_config[0]
    num_neurons = best_config[1]
    bs = best_config[2]
    model = create_model(X_train_scaled.shape[1], num_layers, num_neurons)
    model.compile(optimizer='rmsprop', loss='mean_squared_error', metrics=['acc'])
    model.load_weights("best_model.weights.h5")
    #model = load_model("best_model.h5")
    early_stopping = EarlyStopping(monitor='val_loss', patience=1, restore_best_weights=True)
    history = model.fit(X_train_scaled, y_train, epochs=6, batch_size=bs, validation_data=(X_vali_scaled, y_vali), callbacks=[early_stopping], verbose=0)
    predictions = model.predict(X_test)

    # Save predictions to a CSV file
    file_name = f"pred_F_{cf_index}.csv"
    pd.DataFrame(predictions).to_csv(file_name, index=False)

    val_loss = history.history['val_loss'][-1]
    history_dict = history.history
    acc = history.history['acc']
    val_acc = history.history['val_acc']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    epochs = range(1, len(acc) + 1)

    # "bo" is for "blue dot"
    plt.plot(epochs, loss, color='black', linestyle='solid', linewidth=1 , marker='o', markerfacecolor='black', markersize=4, label='Training loss')
    # b is for "solid blue line"
    plt.plot(epochs, val_loss, color='gray', linestyle='dashed', linewidth=1 , marker='+', markerfacecolor='gray', markersize=4, label='Validation loss')
    plt.title('Training and validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.subplots_adjust(hspace=0.5, wspace=0.3)  # Adjust spacing
    plt.tight_layout()
    plt.show()

#X_train, y_train = normalized_df_fs.iloc[:, 2:17], normalized_df_fs.iloc[:, 22].astype(int)
#X_test, y_test = normalized_df_fs.iloc[:, 2:17], normalized_df_fs.iloc[:, 22].astype(int)
#X_vali, y_vali = normalized_df_fs.iloc[:, 2:17], normalized_df_fs.iloc[:, 22].astype(int)

#=====================================================
#=====================================================
#=====================================================
#=====================================================
# Integrate DLs using Choquet Integral
print(tops_CF)
#print(weights)
mu = weights[[i - 1 for i in tops_CF], 1]
#print(mu)

import pandas as pd
import numpy as np
import shutil

# Assuming 'mu' is a numpy array and 'tops_CF' is a list of indices

# Load the prediction files into a list of DataFrames
prediction_dfs = []
for cf_index in tops_CF:
    file_name = f"pred_F_{cf_index}.csv"
    prediction_dfs.append(pd.read_csv(file_name))

# Concatenate the DataFrames horizontally
predictions_df = pd.concat(prediction_dfs, axis=1)

# Create a Choquet integral function
def choquet_integral(x, weights):
    """
    Calculates the Choquet integral of a vector x with respect to a fuzzy measure (weights).

    Args:
        x: A numpy array representing the input vector.
        weights: A numpy array representing the fuzzy measure.

    Returns:
        The Choquet integral value.
    """

    sorted_indices = np.argsort(x)[::-1]
    sorted_x = x[sorted_indices]
    sorted_weights = weights[sorted_indices]

    integral = 0
    for i in range(len(x)):
        integral += (sorted_weights[i] - sorted_weights[i - 1]) * sorted_x[i]
    return integral

# Apply Choquet integral to each row of the predictions DataFrame
predicted_labels = predictions_df.apply(lambda row: choquet_integral(row.values, mu), axis=1)
predictions_df_Choquet_Integral = np.reshape(predicted_labels, (len(y_test),))

# Print or use the final DataFrame with Choquet integral values
#print(predictions_df_Choquet_Integral)
with open(f'predictions_Choquet_Integral.txt', 'w') as f:
    for f1 in predictions_df_Choquet_Integral:
        f.write(f"  {f1:.7e}  \n")

# Evaluate predictions using external script
outputFileName = f"output_pruning_{pruning_threshold}_k_CF_{k_CF_percent}_k_NC_{k_NC_percent}.txt"
print(outputFileName)

!perl mslr-eval-score-mslr.pl judgements.txt predictions_Choquet_Integral.txt output.txt 0

shutil.copyfile("/content/output.txt", f"/content/{outputFileName}")
print(f"Result of Aggregation by the Choquet Integral:\n")
with open(f"/content/{outputFileName}", "r") as final_output:
    print(final_output.read())

**Step#5: Aggregate DL models' outputs using fuzzy integrals**

In [ ]:
# Integrate DLs using Choquet Integral
print(tops_CF)
#print(weights)
mu = weights[[i - 1 for i in tops_CF], 1]
#print(mu)

import pandas as pd
import numpy as np
import shutil

# Assuming 'mu' is a numpy array and 'tops_CF' is a list of indices

# Load the prediction files into a list of DataFrames
prediction_dfs = []
for cf_index in tops_CF:
    file_name = f"pred_F_{cf_index}.csv"
    prediction_dfs.append(pd.read_csv(file_name))

# Concatenate the DataFrames horizontally
predictions_df = pd.concat(prediction_dfs, axis=1)

# Create a Choquet integral function
def choquet_integral(x, weights):
    """
    Calculates the Choquet integral of a vector x with respect to a fuzzy measure (weights).

    Args:
        x: A numpy array representing the input vector.
        weights: A numpy array representing the fuzzy measure.

    Returns:
        The Choquet integral value.
    """

    sorted_indices = np.argsort(x)[::-1]
    sorted_x = x[sorted_indices]
    sorted_weights = weights[sorted_indices]

    integral = 0
    for i in range(len(x)):
        integral += (sorted_weights[i] - sorted_weights[i - 1]) * sorted_x[i]
    return integral

# Apply Choquet integral to each row of the predictions DataFrame
predicted_labels = predictions_df.apply(lambda row: choquet_integral(row.values, mu), axis=1)
predictions_df_Choquet_Integral = np.reshape(predicted_labels, (len(y_test),))

# Print or use the final DataFrame with Choquet integral values
#print(predictions_df_Choquet_Integral)
with open(f'predictions_Choquet_Integral.txt', 'w') as f:
    for f1 in predictions_df_Choquet_Integral:
        f.write(f"  {f1:.7e}  \n")

# Evaluate predictions using external script
outputFileName = f"output_pruning_{pruning_threshold}_k_CF_{k_CF_percent}_k_NC_{k_NC_percent}.txt"
print(outputFileName)

!perl mslr-eval-score-mslr.pl judgements.txt predictions_Choquet_Integral.txt output.txt 0

shutil.copyfile("/content/output.txt", f"/content/{outputFileName}")
print(f"Result of Aggregation by the Choquet Integral:\n")
with open(f"/content/{outputFileName}", "r") as final_output:
    print(final_output.read())

============================================================

============================================================

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import random
from tensorflow.keras.utils import plot_model
import graphviz  # Import graphviz

def create_model(input_shape, num_layers, num_neurons):
    # Initialize a sequential model
    model = Sequential()
    # Add the first dense layer with L2 regularization and a dropout layer
    model.add(Dense(num_neurons, activation='relu', input_shape=(input_shape,), kernel_regularizer=l2(0.008))) #0.008
    model.add(Dropout(0.5))

    # Add additional layers as specified by num_layers
    for _ in range(num_layers - 1):
        model.add(Dense(num_neurons, activation='relu', kernel_regularizer=l2(0.008)))
        model.add(Dropout(0.5))

    # Add the output layer
    model.add(Dense(1, activation='sigmoid'))

    # Compile the model with Adam optimizer and mean squared error loss
    #model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
    model.compile(optimizer='rmsprop', loss='mean_squared_error', metrics=['acc'])

    return model

# Define hyperparameter ranges
num_layers_range = [2, 3, 4]
num_neurons_range = [4, 8, 16, 32, 64, 128, 256]
batch_size_range = [4, 8, 16, 32, 64, 128, 256]

# Number of iterations for random search
num_iterations = 10

# Initialize variables
best_config = None
best_val_loss = float('inf')
val_losses = []
num_layers_history = []
num_neurons_history = []
batch_size_history = []



def create_model(input_shape, num_layers, num_neurons):
    # Initialize a sequential model
    model = Sequential()
    # Add the first dense layer with L2 regularization and a dropout layer
    model.add(Dense(num_neurons, activation='relu', input_shape=(input_shape,), kernel_regularizer=l2(0.008))) #0.008
    model.add(Dropout(0.5))

    # Add additional layers as specified by num_layers
    for _ in range(num_layers - 1):
        model.add(Dense(num_neurons, activation='relu', kernel_regularizer=l2(0.008)))
        model.add(Dropout(0.5))

    # Add the output layer
    model.add(Dense(1, activation='sigmoid'))

    # Compile the model with Adam optimizer and mean squared error loss
    #model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
    model.compile(optimizer='rmsprop', loss='mean_squared_error', metrics=['acc'])

    return model

# Scale the training and validation data and test data
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_vali_scaled = scaler_X.transform(X_vali)
X_test_scaled = scaler_X.transform(X_test)

# Initialize a MinMaxScaler for later use
scaler = MinMaxScaler(feature_range=(0, 2))

# Initialize lists to store predictions
predictions = []
train_predictions = []

# Define configurations for models: num_layer, num_neuron, batch_size
#config = [(3, 256, 64), (2, 128, 64), (3, 64, 32)]
config = [(3, 256, 64), (2, 128, 64), (3, 64, 32), (3, 256, 128), (3, 128, 128)]
count_of_models = num_iterations

# Initialize lists to store evaluation metrics
p_at_1 = [[] for _ in range(count_of_models)]
p_at_2 = [[] for _ in range(count_of_models)]
p_at_3 = [[] for _ in range(count_of_models)]
p_at_4 = [[] for _ in range(count_of_models)]
p_at_5 = [[] for _ in range(count_of_models)]
p_at_6 = [[] for _ in range(count_of_models)]
p_at_7 = [[] for _ in range(count_of_models)]
p_at_8 = [[] for _ in range(count_of_models)]
map = [[] for _ in range(count_of_models)]
ndcg_at_1 = [[] for _ in range(count_of_models)]
ndcg_at_2 = [[] for _ in range(count_of_models)]
ndcg_at_3 = [[] for _ in range(count_of_models)]
ndcg_at_4 = [[] for _ in range(count_of_models)]
ndcg_at_5 = [[] for _ in range(count_of_models)]
ndcg_at_6 = [[] for _ in range(count_of_models)]
ndcg_at_7 = [[] for _ in range(count_of_models)]
ndcg_at_8 = [[] for _ in range(count_of_models)]
meanndcg = [[] for _ in range(count_of_models)]

# Initialize final metric lists
p_at_1_final = []
p_at_2_final = []
p_at_3_final = []
p_at_4_final = []
p_at_5_final = []
p_at_6_final = []
p_at_7_final = []
p_at_8_final = []
map_final = []
ndcg_at_1_final = []
ndcg_at_2_final = []
ndcg_at_3_final = []
ndcg_at_4_final = []
ndcg_at_5_final = []
ndcg_at_6_final = []
ndcg_at_7_final = []
ndcg_at_8_final = []
meanndcg_final = []

"""
# Generate a sample model to create the architecture visualization
sample_model = create_model(X_train_scaled.shape[1], 3, 32)  # Adjust parameters as needed
# Create the model architecture visualization (optional)
dot = plot_model(sample_model, show_shapes=True, show_layer_names=True)  # Optional: Include layer names
graph = graphviz.Source(dot)  # Use dot.source to get the DOT string
# Create the model architecture visualization (optional)
plot_model(sample_model, to_file='model_architecture.png', show_shapes=True, dpi=1200, show_layer_names=True, show_layer_activations=True)  # Increase DPI for higher resolution
"""

test_predictions = []
for iterat in range(num_iterations):
    # Randomly sample hyperparameters
    num_layers = random.choice(num_layers_range)
    num_neurons = random.choice(num_neurons_range)
    bs = random.choice(batch_size_range)
    print(" **** Model Config[",iterat,"]: num_layers=", num_layers, ", num_neurons=", num_neurons, ", batch_size=", bs)

    # Create and train the model
    model = create_model(X_train_scaled.shape[1], num_layers, num_neurons)
    early_stopping = EarlyStopping(monitor='val_loss', patience=1, restore_best_weights=True)
    history = model.fit(X_train_scaled, y_train, epochs=6, batch_size=bs, validation_data=(X_vali_scaled, y_vali), callbacks=[early_stopping], verbose=0)

    # Record hyperparameters and validation loss
    val_loss = history.history['val_loss'][-1]

    # Update best configuration if necessary
    if val_loss < best_val_loss:
        best_config = (num_layers, num_neurons, bs)
        best_val_loss = val_loss
        # Optionally, save the best model for later use
        #model.save("best_model.h5")


    val_losses.append(val_loss)
    num_layers_history.append(num_layers)
    num_neurons_history.append(num_neurons)
    batch_size_history.append(bs)


    # Make predictions on training and test data
    y_ptrain = model.predict(X_train_scaled)
    y_ptest = model.predict(X_test_scaled)

    # Scale the predictions
    scaled_y_ptest = scaler.fit_transform(y_ptest)
    scaled_y_ptrain = scaler.fit_transform(y_ptrain)

    # Append the scaled predictions to the respective lists
    test_predictions.append(scaled_y_ptest)
    train_predictions.append(scaled_y_ptrain)


# Organize the predictions for each model
models_on_test = [[] for _ in range(num_iterations)]
for i in range(num_iterations):
    for p in test_predictions[i]:
        models_on_test[i].append(p)

models_on_train = [[] for _ in range(num_iterations)]
for i in range(num_iterations):
    for p in train_predictions[i]:
        models_on_train[i].append(p)

"""
# Process the test predictions
with open("/content/test.txt", "r") as infile, open(f"/content/judgements.txt", "w") as outfile:
    for line in infile:
        parts = line.split()
        label = parts[0]
        qid_part = parts[1]
        docid_part = parts[-7]
        inc_part = parts[-4]
        prob_part = parts[-1]

        outfile.write(f"{parts[0]} {qid_part} #docid = {docid_part} inc = {inc_part} prob = {prob_part}\n")
"""

# Evaluate each model's predictions
for number_of_model in range(num_iterations):
    print(number_of_model)
    predicted_labels = models_on_test[number_of_model]
    predicted_labels = np.reshape(predicted_labels, (len(y_test),))
    with open(f'predictions-F1.txt', 'w') as f:
        for f1 in predicted_labels:
            f.write(f"  {f1:.7e}  \n")

    # Evaluate predictions using external script
    !perl mslr-eval-score-mslr.pl judgements.txt predictions-F1.txt output.txt 0
    print(f"result for model({number_of_model}):\n")
    with open(f"/content/output.txt", "r") as final_output:
        print(final_output.read())

    # Read and store evaluation metrics
    df = pd.read_csv('/content/output.txt', delimiter='\t', header=None)
    p_at_1[number_of_model].append(float(df.iloc[1, 1]))
    p_at_2[number_of_model].append(float(df.iloc[1, 2]))
    p_at_3[number_of_model].append(float(df.iloc[1, 3]))
    p_at_4[number_of_model].append(float(df.iloc[1, 4]))
    p_at_5[number_of_model].append(float(df.iloc[1, 5]))
    p_at_6[number_of_model].append(float(df.iloc[1, 6]))
    p_at_7[number_of_model].append(float(df.iloc[1, 7]))
    p_at_8[number_of_model].append(float(df.iloc[1, 8]))
    map[number_of_model].append(float(df.iloc[1, 11]))
    ndcg_at_1[number_of_model].append(float(df.iloc[3, 1]))
    ndcg_at_2[number_of_model].append(float(df.iloc[3, 2]))
    ndcg_at_3[number_of_model].append(float(df.iloc[3, 3]))
    ndcg_at_4[number_of_model].append(float(df.iloc[3, 4]))
    ndcg_at_5[number_of_model].append(float(df.iloc[3, 5]))
    ndcg_at_6[number_of_model].append(float(df.iloc[3, 6]))
    ndcg_at_7[number_of_model].append(float(df.iloc[3, 7]))
    ndcg_at_8[number_of_model].append(float(df.iloc[3, 8]))
    meanndcg[number_of_model].append(float(df.iloc[3, 11]))
#-------------------


    # *********************
    """
    history_dict = history.history
    acc = history.history['acc']
    val_acc = history.history['val_acc']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    epochs = range(1, len(acc) + 1)

    # "bo" is for "blue dot"
    plt.plot(epochs, loss, color='black', linestyle='solid', linewidth=1 , marker='o', markerfacecolor='black', markersize=4, label='Training loss')
    # b is for "solid blue line"
    plt.plot(epochs, val_loss, color='gray', linestyle='dashed', linewidth=1 , marker='+', markerfacecolor='gray', markersize=4, label='Validation loss')
    plt.title('Training and validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()
    # *********************
    """

# Visualize results
plt.figure(figsize=(12, 8))

plt.subplot(1, 3, 1)
plt.scatter(num_layers_history, val_losses)
plt.xlabel("Number of Layers")
plt.ylabel("Validation Loss")
plt.title("Validation Loss vs. Number of Layers")
changed_num_layers_range = [num + 1 for num in num_layers_range]
plt.xticks(num_layers_range, changed_num_layers_range)  # Use original values for plotting, modified values for labels

plt.subplot(1, 3, 2)
plt.scatter(num_neurons_history, val_losses)
plt.xlabel("Number of Neurons")
plt.ylabel("Validation Loss")
plt.title("Validation Loss vs. Number of Neurons")
plt.xticks(num_neurons_range)  # Set integer values for x-axis

plt.subplot(1, 3, 3)
plt.scatter(batch_size_history, val_losses)
plt.xlabel("Batch Size")
plt.ylabel("Validation Loss")
plt.title("Validation Loss vs. Batch Size")
plt.xticks(batch_size_range)  # Set integer values for x-axis
#plt.figure(dpi=1200)
plt.subplots_adjust(hspace=0.5, wspace=0.3)  # Adjust spacing
plt.tight_layout()
plt.show()

# Print the best configuration
print("Best Configuration:", best_config)


num_layers = best_config[0]
num_neurons = best_config[1]
bs = best_config[2]
model = create_model(X_train_scaled.shape[1], num_layers, num_neurons)
early_stopping = EarlyStopping(monitor='val_loss', patience=1, restore_best_weights=True)
history = model.fit(X_train_scaled, y_train, epochs=6, batch_size=bs, validation_data=(X_vali_scaled, y_vali), callbacks=[early_stopping], verbose=0)
val_loss = history.history['val_loss'][-1]
history_dict = history.history
acc = history.history['acc']
val_acc = history.history['val_acc']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(acc) + 1)

# "bo" is for "blue dot"
plt.plot(epochs, loss, color='black', linestyle='solid', linewidth=1 , marker='o', markerfacecolor='black', markersize=4, label='Training loss')
# b is for "solid blue line"
plt.plot(epochs, val_loss, color='gray', linestyle='dashed', linewidth=1 , marker='+', markerfacecolor='gray', markersize=4, label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.subplots_adjust(hspace=0.5, wspace=0.3)  # Adjust spacing
plt.tight_layout()
plt.show()

=============================================================

**Assign weights to rankers**

1.   Calculate each ranker's weight (comapred to relevance labels)
2.   Calculate each ranker's diversity (based on Kendall-Tau Matrix)
3.   Mix these values and assign a weight to each ranker

In [ ]:
import pandas as pd
from scipy.stats import kendalltau

def calculate_kendalltau_vector(df):
    """
    Calculates a vector where the i-th element is the Kendall Tau correlation between
    Ranker_i and Label.

    Args:
        df: A pandas DataFrame with columns 'Label', 'qid', and 'Ranker_1', 'Ranker_2', ..., 'Ranker_25'.

    Returns:
        A numpy array containing the Kendall Tau correlations.
    """

    # Extract the headers of the Ranker columns
    ranker_headers = [col for col in df.columns if col.startswith('Ranker')]

    # Calculate Kendall Tau for each Ranker column
    kendalltau_vector = []
    for ranker_col in ranker_headers:
        kendalltau_coef, _ = kendalltau(df['Label'], df[ranker_col])
        kendalltau_vector.append(kendalltau_coef)

    return np.array(kendalltau_vector)

kendalltau_vector = calculate_kendalltau_vector(df_train)

def calculate_weight_vector(kendalltau_matrix, kendalltau_vector):
    """
    Calculates a weight vector where the i-th element is the fraction:
    kendalltau_vector(1,i)/sum(column_i of the kendallTau matrix).

    Args:
        kendalltau_matrix: A NumPy array representing the Kendall Tau matrix.
        kendalltau_vector: A NumPy array representing the Kendall Tau vector.

    Returns:
        A NumPy array representing the weight vector.
    """

    # Calculate the sum of each column in the Kendall Tau matrix
    column_sums = np.sum(kendalltau_matrix, axis=0)

    # Calculate the weight vector
    weight_vector = kendalltau_vector / column_sums

    return weight_vector
import numpy as np

def normalize_to_0_1(vector):
  """Normalizes a vector to the range [0, 1].

  Args:
    vector: A NumPy array representing the vector to be normalized.

  Returns:
    A NumPy array representing the normalized vector.
  """

  # Find the minimum and maximum values in the vector
  min_val = np.min(vector)
  max_val = np.max(vector)

  # Normalize the vector using min-max normalization
  normalized_vector = (vector - min_val) / (max_val - min_val)

  return normalized_vector

rankers_weight_vector = calculate_weight_vector(kendall_tau_matrix, kendalltau_vector)
normalized_rankers_weight_vector = normalize_to_0_1(rankers_weight_vector)
print(normalized_rankers_weight_vector)

**Set Train, Validation and Test data as all rankers**

In [ ]:
#X_train, y_train = df_train.iloc[:, 2:], df_train.iloc[:, 0].astype(int)
#X_test, y_test = df_test.iloc[:, 2:], df_test.iloc[:, 0].astype(int)
#X_vali, y_vali = df_vali.iloc[:, 2:], df_vali.iloc[:, 0].astype(int)

Train and evaluate Deep Net

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import random
from tensorflow.keras.utils import plot_model
import graphviz  # Import graphviz

# Define hyperparameter ranges
num_layers_range = [2, 3, 4]
num_neurons_range = [32, 64, 128, 256]
batch_size_range = [32, 64, 128]

# Number of iterations for random search
num_iterations = 10

# Initialize variables
best_config = None
best_val_loss = float('inf')
val_losses = []
num_layers_history = []
num_neurons_history = []
batch_size_history = []



def create_model(input_shape, num_layers, num_neurons):
    # Initialize a sequential model
    model = Sequential()
    # Add the first dense layer with L2 regularization and a dropout layer
    model.add(Dense(num_neurons, activation='relu', input_shape=(input_shape,), kernel_regularizer=l2(0.008))) #0.008
    model.add(Dropout(0.5))

    # Add additional layers as specified by num_layers
    for _ in range(num_layers - 1):
        model.add(Dense(num_neurons, activation='relu', kernel_regularizer=l2(0.008)))
        model.add(Dropout(0.5))

    # Add the output layer
    model.add(Dense(1, activation='sigmoid'))

    # Compile the model with Adam optimizer and mean squared error loss
    #model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
    model.compile(optimizer='rmsprop', loss='mean_squared_error', metrics=['acc'])

    return model

# Scale the training and validation data and test data
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_vali_scaled = scaler_X.transform(X_vali)
X_test_scaled = scaler_X.transform(X_test)

# Initialize a MinMaxScaler for later use
scaler = MinMaxScaler(feature_range=(0, 2))

# Initialize lists to store predictions
predictions = []
train_predictions = []

# Define configurations for models: num_layer, num_neuron, batch_size
#config = [(3, 256, 64), (2, 128, 64), (3, 64, 32)]
config = [(3, 256, 64), (2, 128, 64), (3, 64, 32), (3, 256, 128), (3, 128, 128)]
count_of_models = len(config)

# Initialize lists to store evaluation metrics
p_at_1 = [[] for _ in range(count_of_models)]
p_at_2 = [[] for _ in range(count_of_models)]
p_at_3 = [[] for _ in range(count_of_models)]
p_at_4 = [[] for _ in range(count_of_models)]
p_at_5 = [[] for _ in range(count_of_models)]
p_at_6 = [[] for _ in range(count_of_models)]
p_at_7 = [[] for _ in range(count_of_models)]
p_at_8 = [[] for _ in range(count_of_models)]
map = [[] for _ in range(count_of_models)]
ndcg_at_1 = [[] for _ in range(count_of_models)]
ndcg_at_2 = [[] for _ in range(count_of_models)]
ndcg_at_3 = [[] for _ in range(count_of_models)]
ndcg_at_4 = [[] for _ in range(count_of_models)]
ndcg_at_5 = [[] for _ in range(count_of_models)]
ndcg_at_6 = [[] for _ in range(count_of_models)]
ndcg_at_7 = [[] for _ in range(count_of_models)]
ndcg_at_8 = [[] for _ in range(count_of_models)]
meanndcg = [[] for _ in range(count_of_models)]

# Initialize final metric lists
p_at_1_final = []
p_at_2_final = []
p_at_3_final = []
p_at_4_final = []
p_at_5_final = []
p_at_6_final = []
p_at_7_final = []
p_at_8_final = []
map_final = []
ndcg_at_1_final = []
ndcg_at_2_final = []
ndcg_at_3_final = []
ndcg_at_4_final = []
ndcg_at_5_final = []
ndcg_at_6_final = []
ndcg_at_7_final = []
ndcg_at_8_final = []
meanndcg_final = []

# Generate a sample model to create the architecture visualization
sample_model = create_model(X_train_scaled.shape[1], 3, 32)  # Adjust parameters as needed
# Create the model architecture visualization (optional)
dot = plot_model(sample_model, show_shapes=True, show_layer_names=True)  # Optional: Include layer names
graph = graphviz.Source(dot)  # Use dot.source to get the DOT string
# Create the model architecture visualization (optional)
plot_model(sample_model, to_file='model_architecture.png', show_shapes=True, dpi=1200, show_layer_names=True, show_layer_activations=True)  # Increase DPI for higher resolution

for iterat in range(num_iterations):
    # Randomly sample hyperparameters
    num_layers = random.choice(num_layers_range)
    num_neurons = random.choice(num_neurons_range)
    bs = random.choice(batch_size_range)
    print(" **** Model Config[",iterat,"]: num_layers=", num_layers, ", num_neurons=", num_neurons, ", batch_size=", bs)

    # Create and train the model
    model = create_model(X_train_scaled.shape[1], num_layers, num_neurons)
    early_stopping = EarlyStopping(monitor='val_loss', patience=1, restore_best_weights=True)
    history = model.fit(X_train_scaled, y_train, epochs=6, batch_size=bs, validation_data=(X_vali_scaled, y_vali), callbacks=[early_stopping], verbose=0)

    # Record hyperparameters and validation loss
    val_loss = history.history['val_loss'][-1]

    # Update best configuration if necessary
    if val_loss < best_val_loss:
        best_config = (num_layers, num_neurons, bs)
        best_val_loss = val_loss
        # Optionally, save the best model for later use
        #model.save("best_model.h5")


    val_losses.append(val_loss)
    num_layers_history.append(num_layers)
    num_neurons_history.append(num_neurons)
    batch_size_history.append(bs)
    # *********************
    """
    history_dict = history.history
    acc = history.history['acc']
    val_acc = history.history['val_acc']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    epochs = range(1, len(acc) + 1)

    # "bo" is for "blue dot"
    plt.plot(epochs, loss, color='black', linestyle='solid', linewidth=1 , marker='o', markerfacecolor='black', markersize=4, label='Training loss')
    # b is for "solid blue line"
    plt.plot(epochs, val_loss, color='gray', linestyle='dashed', linewidth=1 , marker='+', markerfacecolor='gray', markersize=4, label='Validation loss')
    plt.title('Training and validation loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()
    # *********************
    """

# Visualize results
plt.figure(figsize=(12, 8))

plt.subplot(1, 3, 1)
plt.scatter(num_layers_history, val_losses)
plt.xlabel("Number of Layers")
plt.ylabel("Validation Loss")
plt.title("Validation Loss vs. Number of Layers")
changed_num_layers_range = [num + 1 for num in num_layers_range]
plt.xticks(num_layers_range, changed_num_layers_range)  # Use original values for plotting, modified values for labels

plt.subplot(1, 3, 2)
plt.scatter(num_neurons_history, val_losses)
plt.xlabel("Number of Neurons")
plt.ylabel("Validation Loss")
plt.title("Validation Loss vs. Number of Neurons")
plt.xticks(num_neurons_range)  # Set integer values for x-axis

plt.subplot(1, 3, 3)
plt.scatter(batch_size_history, val_losses)
plt.xlabel("Batch Size")
plt.ylabel("Validation Loss")
plt.title("Validation Loss vs. Batch Size")
plt.xticks(batch_size_range)  # Set integer values for x-axis
#plt.figure(dpi=1200)
plt.subplots_adjust(hspace=0.5, wspace=0.3)  # Adjust spacing
plt.tight_layout()
plt.show()

# Print the best configuration
print("Best Configuration:", best_config)


num_layers = best_config[0]
num_neurons = best_config[1]
bs = best_config[2]
model = create_model(X_train_scaled.shape[1], num_layers, num_neurons)
early_stopping = EarlyStopping(monitor='val_loss', patience=1, restore_best_weights=True)
history = model.fit(X_train_scaled, y_train, epochs=6, batch_size=bs, validation_data=(X_vali_scaled, y_vali), callbacks=[early_stopping], verbose=0)
val_loss = history.history['val_loss'][-1]
history_dict = history.history
acc = history.history['acc']
val_acc = history.history['val_acc']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(acc) + 1)

# "bo" is for "blue dot"
plt.plot(epochs, loss, color='black', linestyle='solid', linewidth=1 , marker='o', markerfacecolor='black', markersize=4, label='Training loss')
# b is for "solid blue line"
plt.plot(epochs, val_loss, color='gray', linestyle='dashed', linewidth=1 , marker='+', markerfacecolor='gray', markersize=4, label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.subplots_adjust(hspace=0.5, wspace=0.3)  # Adjust spacing
plt.tight_layout()
plt.show()
